# 법률 문서 검토 보조 - AI 모델 학습 및 평가

이 노트북은 프로젝트에 포함된 두 개의 머신러닝 모델을 Google Colab에서 학습하고 평가합니다.

1. 문서 유형 분류 모델: 개인정보 동의서, 일반 약관, 주택 임대차, 근로계약
2. 계약 조항 위험 유형 분류 모델: 책임 면제, 해지 제한, 위약금, 임대차 갱신권, 근로 위약, 장시간 근로 등

두 모델 모두 문자 n-gram TF-IDF와 Logistic Regression을 사용합니다. 합성 템플릿 데이터의 내부 평가 결과는 실제 문서에 대한 일반화 성능을 의미하지 않습니다.

## 1. 저장소 복제 및 패키지 설치

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/sam3319/privacy-consent-review-ai.git"
PROJECT_DIR = Path("/content/privacy-consent-review-ai")

if Path("src").exists() and Path("scripts").exists():
    PROJECT_DIR = Path.cwd()
elif not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
sys.path.insert(0, str(PROJECT_DIR.resolve()))
print(PROJECT_DIR.resolve())

## 2. 합성 학습 데이터 생성

In [ ]:
from scripts.generate_training_data import build_dataset as build_document_dataset
from scripts.generate_clause_training_data import build_dataset as build_clause_dataset

document_data_path = build_document_dataset()
clause_data_path = build_clause_dataset()
print("문서 유형 데이터:", document_data_path)
print("조항 위험 데이터:", clause_data_path)

## 3. 문서 유형 분류 모델 학습

총 320건, 4개 클래스의 합성 데이터를 사용합니다. 층화 홀드아웃과 5겹 교차검증 결과를 저장합니다.

In [ ]:
from scripts.train_document_classifier import train_and_save as train_document_model

document_metrics = train_document_model()
print(json.dumps(document_metrics, ensure_ascii=False, indent=2))

## 4. 계약 조항 위험 유형 모델 학습

총 420건, 7개 클래스의 합성 조항 데이터를 사용합니다.

In [ ]:
from scripts.train_clause_classifier import train_and_save as train_clause_model

clause_metrics = train_clause_model()
print(json.dumps(clause_metrics, ensure_ascii=False, indent=2))

## 5. 저장된 모델 파일 확인

In [ ]:
for path in sorted(Path("models").glob("*")):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

## 6. 샘플 문서 추론

In [ ]:
from src.ml_classifier import predict_clause_risks, predict_document_type

sample_paths = [
    "samples/complete_collection_consent.txt",
    "samples/risky_standard_terms_contract.txt",
    "samples/risky_housing_lease.txt",
    "samples/risky_employment_contract.txt",
]

for sample_path in sample_paths:
    text = Path(sample_path).read_text(encoding="utf-8")
    prediction = predict_document_type(text)
    print(Path(sample_path).name, prediction["label"], prediction["probability"])

contract_text = Path("samples/risky_standard_terms_contract.txt").read_text(encoding="utf-8")
print(json.dumps(predict_clause_risks(contract_text), ensure_ascii=False, indent=2))

## 7. 전체 테스트

모델 산출물, 예측 결과, 규칙 엔진 및 Streamlit 흐름을 함께 검증합니다.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## 모델 한계

- 학습 데이터는 개인정보를 포함하지 않는 합성 템플릿 데이터입니다.
- 내부 평가 점수가 높아도 실제 계약서와 OCR 문서에서 같은 성능을 보장하지 않습니다.
- 모델은 문서 유형과 조항 유형을 분류할 뿐 법 위반이나 계약 효력을 판단하지 않습니다.
- 공식 법령에 연결된 규칙 엔진 결과를 법률 검토 근거로 우선 사용합니다.